# 11 — Phase T: Tantivy Per-File Full-Text Search

AI-Lake Phase T embeds a **Tantivy inverted index** inside every Parquet file's `AILK_FTS` section.
This enables **O(log N) keyword search** directly from the AI-Lake file — no external Elasticsearch or Solr needed.

| Feature | BM25 (Phase 5) | **Tantivy FTS (Phase T)** |
|---|---|---|
| Complexity | O(N) brute-force scan | **O(log N) inverted index** |
| Storage overhead | ~50 kB/file (IDF stats) | **~3–4 MB/file (50k docs)** |
| Multi-column indexing | No | **Yes** |
| Tokenizer options | Whitespace | **default / en_stem / whitespace** |
| Legacy compatibility | Always | Fallback to BM25 for old files |

**API summary:**
```python
# Write with Tantivy FTS
w = ailake.TableWriter(path, dim=32, fts_text_columns=['text'], fts_tokenizer='default')
w.write_batch(texts, embeddings)
w.commit()

# Keyword search — Tantivy fast path when FTS blob present
results = ailake.search_text(path, 'rust programming', top_k=5, text_column='text')
```

In [ ]:
import json, math, pathlib, numpy as np
import ailake

PAYLOAD_PATH = pathlib.Path('/data/demo_query.json')
if PAYLOAD_PATH.exists():
    payload  = json.loads(PAYLOAD_PATH.read_text())
    FTS_PATH = payload['table_paths']['fts']
    BM25_PATH= payload['table_paths']['bm25']
    DIM      = payload['dim']
    METRIC   = payload['metric']
else:
    FTS_PATH  = '/tmp/nb11_fts'
    BM25_PATH = '/tmp/nb11_bm25'
    DIM, METRIC = 32, 'cosine'

print(f'FTS table:  {FTS_PATH}')
print(f'BM25 table: {BM25_PATH}')
print(f'DIM={DIM}  METRIC={METRIC}')

## 1. Write a table with Tantivy FTS enabled

Pass `fts_text_columns=['text']` to `TableWriter`. AI-Lake will:
1. Build a **Tantivy inverted index** for the `text` column on every `write_batch`.
2. Serialize the index as `AILK_FTS` bytes (magic `AFTS`, zstd-compressed).
3. Append the blob to each Parquet file's AILK section.

The file remains a valid Parquet + HNSW file — FTS is additive.

In [ ]:
import pathlib, numpy as np

NB_FTS_PATH = '/tmp/nb11_fts_demo'
pathlib.Path(NB_FTS_PATH).mkdir(parents=True, exist_ok=True)

rng = np.random.default_rng(42)

topics = [
    'rust programming systems language memory safety ownership',
    'python machine learning data science numpy pandas',
    'rust async tokio concurrency futures channels',
    'javascript typescript react frontend components',
    'rust memory borrowing lifetimes references',
    'database sql query optimization index btree',
    'vector search approximate nearest neighbor hnsw',
    'distributed computing apache spark hadoop mapreduce',
    'rust cargo crates dependencies ecosystem',
    'deep learning neural network transformer attention',
]

embeddings = rng.standard_normal((len(topics), DIM)).astype(np.float32)
embeddings /= np.linalg.norm(embeddings, axis=1, keepdims=True)

# Enable Tantivy FTS on the 'text' column
w = ailake.TableWriter(
    NB_FTS_PATH,
    dim=DIM,
    metric=METRIC,
    fts_text_columns=['text'],
    fts_tokenizer='default',  # whitespace + lowercase
)
w.write_batch(topics, embeddings.tolist())
snap = w.commit()
print(f'Table written: {len(topics)} rows, snapshot_id={snap}')
print(f'FTS blob (AILK_FTS) embedded in every Parquet file.')

## 2. Tantivy keyword search — O(log N) fast path

`ailake.search_text` detects the `AILK_FTS` section and uses Tantivy's inverted index.
For legacy files without FTS, it falls back to BM25 O(N) scan automatically.

Results use the same `{row_id, distance, file}` shape as vector search.
`distance` is the negated BM25/Tantivy score — **lower = more relevant**.

In [ ]:
query = 'rust async programming'

results = ailake.search_text(
    NB_FTS_PATH,
    query_text=query,
    top_k=5,
    text_column='text',
)

print(f'Query: "{query}"\n')
print(f'Top-{len(results)} Tantivy FTS results:')
for i, r in enumerate(results, 1):
    text_preview = topics[r['row_id']][:60] if r['row_id'] < len(topics) else '?'
    print(f'  [{i}] row_id={r["row_id"]:2d}  score={-r["distance"]:6.4f}  "{text_preview}"')

## 3. Compare: Tantivy vs. BM25

Both return the same `search_text` API. The key difference:
- **Tantivy table**: query hits the inverted index inside each file — O(log N).
- **BM25 table** (legacy): query is scored against every row — O(N).

For 200 rows the latency difference is negligible. At 50 million rows, it's the difference between milliseconds and minutes.

In [ ]:
import time

query = 'rust memory ownership'

# Tantivy path
t0 = time.perf_counter()
fts_results = ailake.search_text(NB_FTS_PATH, query_text=query, top_k=5, text_column='text')
t_fts = time.perf_counter() - t0

# BM25 path — write a tiny legacy table without FTS for comparison
NB_BM25_LEGACY = '/tmp/nb11_bm25_legacy'
pathlib.Path(NB_BM25_LEGACY).mkdir(parents=True, exist_ok=True)
w2 = ailake.TableWriter(NB_BM25_LEGACY, dim=DIM, metric=METRIC, bm25_text_column='text')
w2.write_batch(topics, embeddings.tolist())
w2.commit()

t0 = time.perf_counter()
bm25_results = ailake.search_text(NB_BM25_LEGACY, query_text=query, top_k=5, text_column='text')
t_bm25 = time.perf_counter() - t0

print(f'Query: "{query}"\n')
print(f'Tantivy (O(log N)):  {t_fts*1000:.1f} ms — top result: row_id={fts_results[0]["row_id"] if fts_results else None}')
print(f'BM25    (O(N)):       {t_bm25*1000:.1f} ms — top result: row_id={bm25_results[0]["row_id"] if bm25_results else None}')
print()
print('Top-3 Tantivy:')
for r in fts_results[:3]:
    print(f'  row_id={r["row_id"]}  score={-r["distance"]:.4f}  "{topics[r["row_id"]][:55]}"')
print()
print('Top-3 BM25:')
for r in bm25_results[:3]:
    print(f'  row_id={r["row_id"]}  score={-r["distance"]:.4f}  "{topics[r["row_id"]][:55]}"')

## 4. Multi-column FTS indexing

Pass multiple column names to `fts_text_columns` — Tantivy indexes all of them in the same index.
A query matching any column surfaces the row.

In [ ]:
NB_MULTI_PATH = '/tmp/nb11_fts_multi'
pathlib.Path(NB_MULTI_PATH).mkdir(parents=True, exist_ok=True)

# Two separate columns: short title + longer description
titles = [t.split()[0].capitalize() for t in topics]  # e.g. 'Rust', 'Python', ...
descriptions = topics  # use the full text as description

w3 = ailake.TableWriter(
    NB_MULTI_PATH,
    dim=DIM,
    metric=METRIC,
    fts_text_columns=['title', 'description'],
    fts_tokenizer='default',
)
w3.write_batch(
    descriptions,          # primary text column for embeddings
    embeddings.tolist(),
    extra_columns={
        'title':       titles,
        'description': descriptions,
    },
)
snap = w3.commit()
print(f'Multi-column FTS table: snapshot_id={snap}')

# Search in 'title' column — short tokens
r_title = ailake.search_text(NB_MULTI_PATH, query_text='Rust', top_k=3, text_column='title')
print(f'\nQuery "Rust" in title column ({len(r_title)} hits):')
for r in r_title:
    print(f'  row_id={r["row_id"]}  title="{titles[r["row_id"]]}"  score={-r["distance"]:.4f}')

# Search in 'description' column — full text
r_desc = ailake.search_text(NB_MULTI_PATH, query_text='transformer attention', top_k=3, text_column='description')
print(f'\nQuery "transformer attention" in description column ({len(r_desc)} hits):')
for r in r_desc:
    print(f'  row_id={r["row_id"]}  score={-r["distance"]:.4f}  "{descriptions[r["row_id"]][:60]}"')

## 5. Tantivy query syntax

Tantivy supports standard query syntax:
- `word` — single term
- `"exact phrase"` — phrase query
- `word1 word2` — implicit OR
- `+word` — required term (AND)
- `-word` — exclude term

AI-Lake applies a 3-tier parse fallback so reserved keywords (`AND`, `OR`, `NOT` as literals) never panic:
1. Full Tantivy parse.
2. Strip special chars, retry.
3. Per-word phrase quoting (`"go" "AND" "OR"`).

In [ ]:
queries = [
    'rust',                         # single term
    'rust async',                   # OR (both terms boost)
    'rust concurrency',             # OR
    'go AND OR',                    # reserved keywords — fallback handles this
    'c++ templates',                # special chars — fallback strips +
    '"memory safety"',              # phrase query
]

for q in queries:
    results = ailake.search_text(NB_FTS_PATH, query_text=q, top_k=1, text_column='text')
    top = f'row_id={results[0]["row_id"]} score={-results[0]["distance"]:.4f}' if results else 'no hits'
    print(f'  {q!r:30s} → {top}')

## 6. FTS + HNSW hybrid

FTS and vector search coexist in the same file. Use each independently or combine results:
- **FTS**: exact keyword match, no embedding required — great for named entities, product codes, exact phrases.
- **HNSW**: semantic similarity — great for paraphrase, conceptual questions.
- **Combined**: intersect or re-rank — e.g. FTS to pre-filter, HNSW to rank.

In [ ]:
query_text = 'tokio async'
query_vec  = embeddings[2].tolist()  # 'rust async tokio concurrency…'

# FTS — keyword precision
fts_hits = ailake.search_text(NB_FTS_PATH, query_text=query_text, top_k=3, text_column='text')

# HNSW — semantic recall
vec_hits = ailake.search(NB_FTS_PATH, query_vec, top_k=3)

print(f'FTS results for "{query_text}":')
for r in fts_hits:
    print(f'  row_id={r["row_id"]:2d}  score={-r["distance"]:.4f}  "{topics[r["row_id"]][:60]}"')

print(f'\nHNSW results (semantic):')
for r in vec_hits:
    print(f'  row_id={r["row_id"]:2d}  dist={r["distance"]:.4f}  "{topics[r["row_id"]][:60]}"')

# Simple intersection re-rank: rows in both sets, sorted by FTS score
fts_ids  = {r['row_id'] for r in fts_hits}
vec_ids  = {r['row_id'] for r in vec_hits}
overlap  = fts_ids & vec_ids
print(f'\nRows in both (FTS ∩ HNSW): {sorted(overlap)}')
print('These rows match both keyword and semantic criteria — highest recall.')

## 7. Pre-built fixture table (`ailake_fts`)

The `init_demo.py` startup script wrote 200 rows to `FTS_PATH` with `fts_text_columns=['text']`.
This table is always available in the demo environment.

In [ ]:
if PAYLOAD_PATH.exists():
    fixture_results = ailake.search_text(
        FTS_PATH,
        query_text='vector search approximate nearest neighbor',
        top_k=5,
        text_column='text',
    )
    print(f'Fixture FTS table ({FTS_PATH}):')
    print(f'{len(fixture_results)} result(s) for "vector search approximate nearest neighbor"')
    for r in fixture_results:
        print(f'  row_id={r["row_id"]:3d}  score={-r["distance"]:6.4f}  file=...{r["file"][-28:]}')
else:
    print('Fixture not available — run init_demo.py first or use the notebook cells above.')
    print('All cells above work without the fixture (they create /tmp tables inline).')

## 8. Storage layout

The `AILK_FTS` blob is stored between the Parquet data and the AILK HNSW section:

```
┌─────────────────────────────────────────────────┐
│  PAR1 header                                    │
│  Parquet row groups (text + embedding columns)  │
│  AILK HNSW section  (magic AILK, bincode)       │
│  AILK FTS section   (magic AFTS, zstd Tantivy)  │  ← Phase T
│  Parquet footer (+ ailake.fts_offset KV)        │
│  PAR1 magic                                     │
└─────────────────────────────────────────────────┘
```

Readers without the AI-Lake SDK stop at `PAR1` and see a valid Parquet file.  
BM25-only readers (Phase 5) ignore `AFTS` and use BM25 stats from `metadata/`.  
Phase T readers detect `AFTS` and use Tantivy O(log N) instead.

Typical sizes (50k docs):
- HNSW section: ~15 MB
- FTS blob:     ~3–4 MB (zstd-compressed Tantivy index)
- Ratio: FTS adds ~20–25% overhead on top of HNSW.